In [2]:
import plotly.graph_objects as go
import numpy as np

# 修正后的损失函数（更贴近真实深度学习的损失曲面：平滑+合理的坑洼分布）
def loss_function(x, y):
    # 全局最小值（最深处的坑）：(0, 0)，损失≈0
    global_min = -2 * np.exp(-0.5*(x**2 + y**2)) * (np.cos(3*x)**2 + np.cos(3*y)**2)

    # 3个典型的局部最小值（浅坑，解较差）
    local_min1 = -1.2 * np.exp(-0.5*((x-2)**2 + (y-2)**2)) * (np.cos(2*(x-2))**2 + np.cos(2*(y-2))**2)
    local_min2 = -1.0 * np.exp(-0.5*((x+2)**2 + (y+2)**2)) * (np.cos(2*(x+2))**2 + np.cos(2*(y+2))**2)
    local_min3 = -0.8 * np.exp(-0.5*((x-2)**2 + (y+2)**2)) * (np.cos(2*(x-2))**2 + np.cos(2*(y+2))**2)

    # 平缓的背景（避免陡峭悬崖掩盖坑洼）
    background = 0.1 * (x**2 + y**2)

    # 总损失：确保全局最小值<局部最小值，曲面平滑
    loss = -(global_min + local_min1 + local_min2 + local_min3) + background + 0.5
    return loss

# 生成网格点（更密集，曲面更平滑）
x = np.linspace(-4, 4, 100)
y = np.linspace(-4, 4, 100)
X, Y = np.meshgrid(x, y)
Z = loss_function(X, Y)

# 标记关键点位（全局/局部最小值）
global_min_x, global_min_y = 0, 0
local_min_pts = [(2, 2), (-2, -2), (2, -2)]  # 3个局部最小值坐标

# 构建交互式3D图
fig = go.Figure()

# 1. 绘制损失曲面（可拖拽旋转）
fig.add_trace(go.Surface(
    x=X, y=Y, z=Z,
    colorscale='Viridis',  # 配色更自然
    opacity=0.8,           # 半透明，能看到标记点
    name='损失曲面'
))

# 2. 标记全局最小值（红色大圆点）
fig.add_trace(go.Scatter3d(
    x=[global_min_x], y=[global_min_y], z=[loss_function(global_min_x, global_min_y)],
    mode='markers',
    marker=dict(size=8, color='red', symbol='circle'),
    name='全局最小值（最优解）'
))

# 3. 标记局部最小值（橙色圆点）
local_x = [p[0] for p in local_min_pts]
local_y = [p[1] for p in local_min_pts]
local_z = [loss_function(p[0], p[1]) for p in local_min_pts]
fig.add_trace(go.Scatter3d(
    x=local_x, y=local_y, z=local_z,
    mode='markers',
    marker=dict(size=6, color='orange', symbol='circle'),
    name='局部最小值（差解）'
))

# 设置图的样式（中文显示+交互优化）
fig.update_layout(
    title=dict(text='交互式深度学习损失曲面（可拖拽旋转/缩放）', font=dict(size=16)),
    scene=dict(
        xaxis_title='参数1',
        yaxis_title='参数2',
        zaxis_title='损失值',
        xaxis=dict(range=[-4, 4]),
        yaxis=dict(range=[-4, 4]),
        zaxis=dict(range=[np.min(Z)-0.5, np.max(Z)+0.5])
    ),
    width=1000,
    height=800,
    legend=dict(font=dict(size=12))
)

# 显示交互式图（运行后会弹出浏览器窗口/Notebook内交互）
fig.show()

In [4]:
import plotly.graph_objects as go
import numpy as np

# 复用修正后的损失函数
def loss_function(x, y):
    global_min = -2 * np.exp(-0.5*(x**2 + y**2)) * (np.cos(3*x)**2 + np.cos(3*y)**2)
    local_min1 = -1.2 * np.exp(-0.5*((x-2)**2 + (y-2)**2)) * (np.cos(2*(x-2))**2 + np.cos(2*(y-2))**2)
    local_min2 = -1.0 * np.exp(-0.5*((x+2)**2 + (y+2)**2)) * (np.cos(2*(x+2))**2 + np.cos(2*(y+2))**2)
    local_min3 = -0.8 * np.exp(-0.5*((x-2)**2 + (y+2)**2)) * (np.cos(2*(x-2))**2 + np.cos(2*(y+2))**2)
    background = 0.1 * (x**2 + y**2)
    loss = -(global_min + local_min1 + local_min2 + local_min3) + background + 0.5
    return loss

# 计算梯度（模拟反向传播）
def compute_gradient(x, y, h=1e-5):
    dx = (loss_function(x+h, y) - loss_function(x-h, y)) / (2*h)
    dy = (loss_function(x, y+h) - loss_function(x, y-h)) / (2*h)
    return np.array([dx, dy])

# 模拟训练（返回参数轨迹和损失轨迹）
def simulate_training(init_x, init_y, total_steps=80):
    # 轨迹存储
    no_warmup_traj = [(init_x, init_y)]
    warmup_traj = [(init_x, init_y)]
    no_warmup_loss = [loss_function(init_x, init_y)]
    warmup_loss = [loss_function(init_x, init_y)]

    # 初始参数
    x1, y1 = init_x, init_y  # 无warm-up
    x2, y2 = init_x, init_y  # 有warm-up

    for step in range(total_steps):
        # ========== 1. 无warm-up：全程大学习率 ==========
        lr_no_warmup = 0.6  # 直接用大学习率
        grad1 = compute_gradient(x1, y1)
        x1 -= lr_no_warmup * grad1[0]
        y1 -= lr_no_warmup * grad1[1]
        no_warmup_traj.append((x1, y1))
        no_warmup_loss.append(loss_function(x1, y1))

        # ========== 2. 有warm-up：先小后大再余弦降 ==========
        # warm-up阶段（前20步）：线性升学习率
        if step < 20:
            lr_warmup = 0.05 + (0.5 - 0.05) * (step / 20)
        # 余弦退火阶段（后60步）：缓慢降学习率
        else:
            anneal_step = step - 20
            anneal_total = total_steps - 20
            lr_warmup = 0.01 + 0.5 * (0.5 - 0.01) * (1 + np.cos(np.pi * anneal_step / anneal_total))

        grad2 = compute_gradient(x2, y2)
        x2 -= lr_warmup * grad2[0]
        y2 -= lr_warmup * grad2[1]
        warmup_traj.append((x2, y2))
        warmup_loss.append(loss_function(x2, y2))

    return {
        "no_warmup": (np.array(no_warmup_traj), no_warmup_loss),
        "warmup": (np.array(warmup_traj), warmup_loss)
    }

# 生成训练数据（初始参数选在“悬崖边”，模拟模型初始化的随机位置）
init_x, init_y = 3.5, 3.5
traj_data = simulate_training(init_x, init_y)

# 提取轨迹
no_warmup_traj, no_warmup_loss = traj_data["no_warmup"]
warmup_traj, warmup_loss = traj_data["warmup"]

# 生成损失曲面背景
x = np.linspace(-4, 4, 50)
y = np.linspace(-4, 4, 50)
X, Y = np.meshgrid(x, y)
Z = loss_function(X, Y)

# 构建交互式对比图
fig = go.Figure()

# 1. 绘制损失曲面背景
fig.add_trace(go.Contour(
    x=x, y=y, z=Z,
    colorscale='Viridis',
    opacity=0.7,
    name='损失等高线',
    showscale=False
))

# 2. 无warm-up的轨迹（红色，带箭头）
fig.add_trace(go.Scatter(
    x=no_warmup_traj[:,0], y=no_warmup_traj[:,1],
    mode='lines+markers+text',
    line=dict(color='red', width=2),
    marker=dict(size=4, color='red'),
    text=[f'{i}' for i in range(len(no_warmup_traj))],
    textposition='top center',
    name='无warm-up（直接大学习率）'
))

# 3. 有warm-up的轨迹（蓝色，带箭头）
fig.add_trace(go.Scatter(
    x=warmup_traj[:,0], y=warmup_traj[:,1],
    mode='lines+markers+text',
    line=dict(color='blue', width=2),
    marker=dict(size=4, color='blue'),
    text=[f'{i}' for i in range(len(warmup_traj))],
    textposition='bottom center',
    name='有warm-up（先小后大再降）'
))

# 标记全局最小值
fig.add_trace(go.Scatter(
    x=[0], y=[0],
    mode='markers',
    marker=dict(size=10, color='green', symbol='star'),
    name='全局最小值'
))

# 设置图样式
fig.update_layout(
    title=dict(text='warm-up vs 无warm-up 训练轨迹对比（交互式）', font=dict(size=16)),
    xaxis_title='参数1',
    yaxis_title='参数2',
    width=900,
    height=800,
    legend=dict(font=dict(size=12)),
    xaxis=dict(range=[-4, 4]),
    yaxis=dict(range=[-4, 4])
)

# 显示图
fig.show()

# ========== 额外：绘制损失变化曲线 ==========
fig_loss = go.Figure()
fig_loss.add_trace(go.Scatter(
    x=list(range(len(no_warmup_loss))), y=no_warmup_loss,
    mode='lines', line=dict(color='red', width=2),
    name='无warm-up 损失'
))
fig_loss.add_trace(go.Scatter(
    x=list(range(len(warmup_loss))), y=warmup_loss,
    mode='lines', line=dict(color='blue', width=2),
    name='有warm-up 损失'
))
fig_loss.update_layout(
    title='训练过程中损失变化',
    xaxis_title='训练步数',
    yaxis_title='损失值',
    width=900,
    height=400,
    legend=dict(font=dict(size=12))
)
fig_loss.show()